<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Deploy Docker Containers on FABRIC

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**Welcome!** This notebook demonstrates how to deploy and manage Docker containers on FABRIC nodes. You will learn how to use FABRIC's Docker-enabled images, start containers with Docker Compose via post boot tasks, run containers from the command line, and execute commands inside running containers.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Create a FABRIC node with a **Docker-enabled image** (`docker_rocky_8`)
2. Use **post boot tasks** to enable Docker and start containers via Docker Compose at slice creation
3. Use the **`{{ _self_.image }}` template** to pass the node's image type to configuration scripts
4. Run Docker containers from the **command line** on a FABRIC node
5. **Execute commands** inside a running Docker container
6. List and inspect running containers

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Be familiar with post boot tasks (see [Post Boot Tasks](../post_boot_tasks/post_boot_tasks.ipynb))

**Tip:** This notebook uses a 100 GB disk because Docker images and containers require additional storage. The `docker_rocky_8` image comes with Docker pre-installed but not yet enabled.

</div>

## Background: Docker on FABRIC

Docker containers let you package and run applications in isolated environments on your FABRIC nodes. This is useful for:

- **Reproducible experiments** -- share a Docker image and anyone can replicate your setup
- **Software isolation** -- run different software stacks on the same node without conflicts
- **Quick deployment** -- pull pre-built images instead of installing software from scratch

FABRIC provides Docker-ready VM images (like `docker_rocky_8`) that have Docker pre-installed. The workflow is:


This notebook uses two approaches to start containers:
1. **Docker Compose** (via post boot tasks) -- for multi-container setups defined in a `docker-compose.yml` file
2. **`docker run`** (via command line) -- for quickly starting individual containers

## What We're Building

In this notebook we will create a single node where we will install and run Docker containers.

<img src="./figs/slice_topology.png" width="40%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Create the Slice with Docker Post Boot Tasks

We create a slice with a single node using the `docker_rocky_8` image and 100 GB of disk. The post boot tasks:
1. Upload `node_tools/` containing the `enable_docker.sh` script
2. Run `enable_docker.sh` with the image name template to start the Docker daemon
3. Upload `docker_containers/` containing a Docker Compose project
4. Start the containers with `docker compose up -d`

<div class="fab-info">

**Template note:** The `{{ _self_.image }}` template resolves to the node's image name (e.g., `docker_rocky_8`). This allows the `enable_docker.sh` script to configure Docker differently depending on the OS image.

</div>

In [ ]:
# Create a new slice
slice = fablib.new_slice(name="MySlice")

# Add a node with a Docker-enabled image and extra disk space for containers
node = slice.add_node(name="Node1", disk=100, image='docker_rocky_8')

# Post boot task 1: Upload the node_tools directory (contains enable_docker.sh)
node.add_post_boot_upload_directory('node_tools','.')

# Post boot task 2: Enable Docker using the helper script.
# The {{ _self_.image }} template passes the node's image name to the script.
node.add_post_boot_execute('node_tools/enable_docker.sh {{ _self_.image }} ')

# Post boot task 3: Upload Docker Compose project files
node.add_post_boot_upload_directory('docker_containers','.')

# Post boot task 4: Start the containers in detached mode using Docker Compose
node.add_post_boot_execute('cd docker_containers/fabric_multitool ; docker compose up -d ')

# Submit the slice -- provisions the node and runs all post boot tasks
slice.submit();

<div class="fab-success">

**What just happened?** FABRIC provisioned a node with the Docker Rocky 8 image, enabled the Docker daemon, and started your containers via Docker Compose -- all automatically during the post boot phase.

</div>

## Step 3: Run a Docker Container from the Command Line

In addition to using Docker Compose via post boot tasks, you can also run containers directly from the command line on an active node. Here we start a container using `docker run`.

In [ ]:
# Get the node reference
node = slice.get_node('Node1')

# Run a container in detached (-d) and interactive (-it) mode.
# --name assigns a friendly name for easy reference later.
stdout, stderr = node.execute("docker run -d -it "
                                "--name fabric_command_line "
                                "fabrictestbed/slice-vm-rocky8-multitool:0.0.1 "
                                , output_file=f"{node.get_name()}.log");

## Step 4: View Running Containers

Use `docker ps -a` to list all containers (both running and stopped) on the node. You should see the Docker Compose container and the command-line container.

In [ ]:
# List all Docker containers on the node (running and stopped)
stdout, stderr = node.execute('docker ps -a')

## Step 5: Execute a Command Inside a Container

Use `docker exec` to run commands inside a running container. This is useful for inspecting the container's network configuration, running tests, or interacting with containerized services.

In [ ]:
# Execute 'ip addr list' inside the container to see its network interfaces
stdout, stderr = node.execute('docker exec fabric_command_line ip addr list')

## Step 6: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- leaving slices running unnecessarily prevents other researchers from using those resources. Deleting the slice also stops and removes all Docker containers on the node.

</div>

In [ ]:
# Delete the slice and release all resources
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `docker: command not found` | Node was not created with a Docker image | Use `image='docker_rocky_8'` in `add_node()` |
| `Cannot connect to the Docker daemon` | Docker service not started | Run `sudo systemctl start docker` on the node; check `enable_docker.sh` |
| `docker compose` not recognized | Older Docker version | Try `docker-compose` (hyphenated) instead of `docker compose` |
| Container exits immediately | Container has no long-running process | Use `-it` flags to keep it running, or run a service that stays in the foreground |
| `No space left on device` | Disk too small for Docker images | Increase disk size in `add_node(disk=200)` |
| Image pull fails | Network connectivity issue | Check if the node has internet access; try pulling the image manually |
| `enable_docker.sh` fails | Script not in `node_tools` directory | Ensure the `node_tools` folder exists in the same directory as this notebook |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `slice.add_node(name, disk, image)` | Add a compute node with specific disk and image | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `node.add_post_boot_upload_directory(local, remote)` | Queue a directory upload for post boot | [add_post_boot_upload_directory](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_post_boot_upload_directory) |
| `node.add_post_boot_execute(command)` | Queue a command for post boot execution | [add_post_boot_execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_post_boot_execute) |
| `slice.submit()` | Submit the slice for provisioning | [submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) |
| `slice.get_node(name)` | Get a specific node object by name | [get_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_node) |
| `node.execute(command)` | Execute a shell command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

Now that you know how to deploy Docker containers on FABRIC, explore these related notebooks:

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **Post Boot Tasks** | [post_boot_tasks](../post_boot_tasks/post_boot_tasks.ipynb) | Basics of automating node configuration at boot time |
| **Post Boot Templates** | [post_boot_task_templates](../post_boot_task_templates/post_boot_task_templates.ipynb) | Use Jinja2 templates for dynamic post boot configuration |
| **Customizing Nodes** | [customizing_nodes](../customizing_nodes/customizing_nodes.ipynb) | Set site, cores, RAM, disk, and OS image |
| **Upload and Execute** | [upload_and_execute](../upload_and_execute/upload_and_execute.ipynb) | Manually upload and run scripts on FABRIC nodes |
| **Parallel Configuration** | [parallel_config](../parallel_config/parallel_config.ipynb) | Configure multiple nodes simultaneously using threads |